# Analyse du graphe d'appels et détection des composantes fortement connexes (SCC)

Ce carnet explore les fonctionnalités avancées d'analyse de code d'UnifyWeaver :

- **Construction du graphe d'appels** - Création de graphes de dépendances à partir de code Prolog
- **Détection des SCC** - Recherche de composantes fortement connexes (récursion mutuelle)
- **Analyse des motifs** - Compréhension des motifs de récursion
- **Visualisation des dépendances** - Représentation visuelle des relations entre prédicats

## Objectifs d'apprentissage

- Comprendre comment UnifyWeaver analyse la structure du code
- Construire et inspecter des graphes d'appels
- Détecter la récursion mutuelle à l'aide de l'algorithme de Tarjan
- Visualiser les dépendances de code

## Configuration

Charger UnifyWeaver et les modules d'analyse.

In [ ]:
% Load initialization
['../init'].

% Load analysis modules
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Exemple 1 : Graphe d'appels simple

Commençons par un prédicat simple et construisons son graphe d'appels.

In [ ]:
% Define ancestor predicate
:- dynamic ancestor/2.
:- dynamic parent/2.

% Parent facts
parent(abraham, isaac).
parent(isaac, jacob).

% Ancestor rules
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### Construire le graphe d'appels

In [ ]:
% Build call graph for ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Analyser les dépendances

In [ ]:
% Get all dependencies of ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Check if self-recursive
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## Exemple 2 : Détection de récursion mutuelle

Détectons maintenant la récursion mutuelle avec l'exemple pair/impair.

In [ ]:
% Define mutually recursive predicates
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### Construire le graphe d'appels pour les deux prédicats

In [ ]:
% Build call graph for both predicates
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Trouver les composantes fortement connexes (SCC)

In [ ]:
% Rebuild the graph because variables do not persist between notebook cells
build_call_graph([is_even/1, is_odd/1], _Graph),
% Find SCCs using Tarjan's algorithm
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### Vérifier si la SCC est triviale

In [ ]:
% Rebuild the derived values so this cell also runs independently
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Check each SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## Exemple 3 : Graphe d'appels complexe

Analysons un système plus complexe comportant plusieurs prédicats.

In [ ]:
% Define a small program with multiple predicates
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent uses parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: same parent, different children
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: parents are siblings
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### Construire le graphe d'appels complet

In [ ]:
% Build call graph for all predicates
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Trouver des groupes de prédicats

Trouver le groupe de prédicats mutuellement récursifs contenant un prédicat de départ.

In [ ]:
% Find the mutually recursive group containing cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## Exemple 4 : Détection de motifs

Utilisons les comparateurs de motifs pour analyser les types de récursion.

In [ ]:
% Define various recursive patterns
:- dynamic count/3.     % Tail recursive
:- dynamic factorial/2. % Linear recursive
:- dynamic fib/2.       % Tree recursive (or linear if detected)

% Tail recursive count
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Linear recursive factorial
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (can be detected as linear or tree)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### Détecter la récursion terminale

In [ ]:
% Check if count/3 is tail recursive
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### Détecter la récursion linéaire

In [ ]:
% Check if factorial/2 is linear recursive
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### Compter les appels récursifs

In [ ]:
% Count recursive calls in fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## Visualisation au format DOT

Générons une représentation Graphviz DOT de notre graphe d'appels.

In [ ]:
% Helper to generate DOT format
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Generate DOT for even/odd graph
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### Enregistrer le fichier DOT

In [ ]:
% Rebuild the DOT source because variables do not persist between cells
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## Exercice : Analysez votre propre code

Essayez de définir vos propres prédicats et de les analyser !

In [ ]:
% Define your predicates here
% Then build call graphs, find SCCs, and detect patterns

% Example:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## Résumé

Dans ce carnet, vous avez appris :

✅ Comment construire des graphes d'appels à partir de code Prolog

✅ Comment détecter les composantes fortement connexes (SCC) pour la récursion mutuelle

✅ Comment utiliser les comparateurs de motifs pour classifier les types de récursion

✅ Comment analyser les dépendances de prédicats

✅ Comment visualiser les graphes d'appels au format DOT

## Sujets avancés

Pour aller plus loin dans l'analyse :

- **Tri topologique** : Utiliser `topological_order/2` pour ordonner les SCC par dépendances
- **Comparateurs de motifs personnalisés** : Écrire vos propres prédicats de détection de motifs
- **Extraction de motif d'accumulateur** : Utiliser `extract_accumulator_pattern/2` pour une analyse détaillée
- **Interdire la récursion linéaire** : Utiliser `forbid_linear_recursion/1` pour forcer d'autres stratégies de compilation

## Références et fichiers associés

- Chapitre 10 : Introspection et théorie Prolog
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`